In [ ]:
from pathlib import Path
import json
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
SEED = 42
tf.keras.utils.set_random_seed(SEED)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32


In [ ]:
import os
import sys
import zipfile
DRIVE_FILE_ID = '1ZPNue82UJ6ySev2IBQnlvqRSbZrITkob'
ARCHIVE_PATH = Path('/content/plant_disease_dataset.zip')
EXTRACT_DIR = Path('/content/plant_disease_extracted')

try:
    import gdown
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
    import gdown

if not ARCHIVE_PATH.exists():
    downloaded = gdown.download(id=DRIVE_FILE_ID, output=str(ARCHIVE_PATH), quiet=False)
    if not downloaded or not ARCHIVE_PATH.exists() or ARCHIVE_PATH.stat().st_size < 1_000_000:
        raise RuntimeError('Dataset download failed. Check that the Drive file is shared as Anyone with the link → Viewer, then rerun this cell.')
    print(f'Downloaded {ARCHIVE_PATH.stat().st_size / (1024**3):.2f} GiB')

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    bad_member = next((name for name in archive.namelist() if Path(name).is_absolute() or '..' in Path(name).parts), None)
    if bad_member:
        raise ValueError(f'Unsafe path in dataset archive: {bad_member}')
    archive.extractall(EXTRACT_DIR)

def find_dataset_root(base):
    for current, dirs, _ in os.walk(base):
        if {'train', 'val', 'test'}.issubset(dirs):
            return Path(current)
    return None

DATA_DIR = find_dataset_root(EXTRACT_DIR)
if DATA_DIR is None:
    raise FileNotFoundError('Could not find train/val/test folders in the ZIP. Check the archive contents.')
print('Dataset folders found at:', DATA_DIR)

splits = ('train', 'val', 'test')
class_sets = {}
for split in splits:
    split_dir = DATA_DIR / split
    if not split_dir.is_dir():
        raise FileNotFoundError(f'Missing split folder: {split_dir.resolve()}')
    class_sets[split] = {p.name for p in split_dir.iterdir() if p.is_dir()}
    print(f'{split}: {len(class_sets[split])} classes')
if not class_sets['train']:
    raise ValueError('No class folders found inside dataset/train')
for split in ('val', 'test'):
    if class_sets[split] != class_sets['train']:
        raise ValueError(f'{split} class folders do not match train. Missing: {sorted(class_sets["train"] - class_sets[split])}; extra: {sorted(class_sets[split] - class_sets["train"])}')
CLASS_NAMES = sorted(class_sets['train'])
print(f'Classes ({len(CLASS_NAMES)}):', CLASS_NAMES)


In [ ]:
import random

MAX_TRAIN_PER_CLASS = 200
MAX_VAL_PER_CLASS = 100
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}
AUTOTUNE = tf.data.AUTOTUNE

def load_split(split, shuffle):
    paths, labels = [], []
    for class_id, class_name in enumerate(CLASS_NAMES):
        files = sorted(p for p in (DATA_DIR / split / class_name).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
        limit = MAX_TRAIN_PER_CLASS if split == 'train' else (MAX_VAL_PER_CLASS if split == 'val' else 0)
        if limit and len(files) > limit:
            files = random.Random(SEED + class_id + (100000 if split == 'val' else 0)).sample(files, limit)
        paths.extend(str(p) for p in files)
        labels.extend([class_id] * len(files))
    if not paths:
        raise ValueError(f'No supported images found in {DATA_DIR / split}')
    print(f'{split}: {len(paths):,} images')
    ds = tf.data.Dataset.from_tensor_slices((paths, np.asarray(labels, dtype=np.int32)))
    if shuffle:
        ds = ds.shuffle(min(len(paths), 10000), seed=SEED, reshuffle_each_iteration=True)
    def decode_image(path, label):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)
        return image, label
    return ds.map(decode_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = load_split('train', True)
val_ds = load_split('val', False)
test_ds = load_split('test', False)


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.1),
], name='data_augmentation')

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.summary()


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.3),
]
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
history_head = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=callbacks)


In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=3, callbacks=callbacks)


In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
probabilities = model.predict(test_ds, verbose=1)
y_pred = np.argmax(probabilities, axis=1)
y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
print(f'Test accuracy: {test_accuracy:.4f}')
print(classification_report(y_true, y_pred, labels=range(len(CLASS_NAMES)), target_names=CLASS_NAMES, zero_division=0, digits=3))


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
fig, ax = plt.subplots(figsize=(max(10, len(CLASS_NAMES) * 0.35), max(8, len(CLASS_NAMES) * 0.3)))
image = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=90, fontsize=6)
ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES, fontsize=6)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Test set confusion matrix')
fig.colorbar(image, ax=ax)
fig.tight_layout()
plt.show()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = Path('/content/runs/mobilenetv2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save(OUTPUT_DIR / 'model.keras')
(OUTPUT_DIR / 'class_names.json').write_text(json.dumps(CLASS_NAMES, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps({
    'test_loss': float(test_loss),
    'test_accuracy': float(test_accuracy),
    'num_classes': len(CLASS_NAMES),
}, indent=2), encoding='utf-8')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/plant_disease_runs/mobilenetv2')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
import shutil
for artifact in OUTPUT_DIR.iterdir():
    if artifact.is_file():
        shutil.copy2(artifact, DRIVE_OUTPUT_DIR / artifact.name)
print(f'Saved model and metadata to {DRIVE_OUTPUT_DIR}')
